
# Bronze Profiling: Sector and Regional Budget

Profile the 2024–2025 Sector and Regional budget Bronze tables before Silver.

**Check:** years, units, reporting statuses, totals, duplicates, blank and missing values, and quality flags.

> Profiling only. Bronze remains unchanged. All queries are read-only.

## 1. Unified Profiling Check

Execution date: **2026-10-07**

One row per assigned table (six tables). Sector amounts are in **million PHP** and sector shares are in **percent**. Regional amounts are in **thousand PHP**.


In [0]:
%sql
-- ============================================================
-- Unified profiling: 6 assigned Sector & Regional Bronze tables
-- ============================================================

-- Table name, category and expected unit are labelled by hand
WITH all_tables AS (
    SELECT 'regional_2024_regional_cells' AS table_name, 'Regional' AS category, 'Thousand PHP' AS expected_unit, record_id, dimensions_json, raw_value_json, parsed_numeric FROM pondolens.bronze.regional_2024_regional_cells
    UNION ALL
    SELECT 'regional_2025_regional_cells', 'Regional', 'Thousand PHP', record_id, dimensions_json, raw_value_json, parsed_numeric FROM pondolens.bronze.regional_2025_regional_cells
    UNION ALL
    SELECT 'sector_2024_sector_amounts', 'Sector', 'Million PHP', record_id, dimensions_json, raw_value_json, parsed_numeric FROM pondolens.bronze.sector_2024_sector_amounts
    UNION ALL
    SELECT 'sector_2024_sector_shares', 'Sector', 'Percentage / Share', record_id, dimensions_json, raw_value_json, parsed_numeric FROM pondolens.bronze.sector_2024_sector_shares
    UNION ALL
    SELECT 'sector_2025_sector_amounts', 'Sector', 'Million PHP', record_id, dimensions_json, raw_value_json, parsed_numeric FROM pondolens.bronze.sector_2025_sector_amounts
    UNION ALL
    SELECT 'sector_2025_sector_shares', 'Sector', 'Percentage / Share', record_id, dimensions_json, raw_value_json, parsed_numeric FROM pondolens.bronze.sector_2025_sector_shares
),

-- Helper columns. Each check reads the value INSIDE the JSON, not the whole JSON text.
-- table_year   = the year in the table name (what the table claims to be)
-- years_in_row = every year (20xx) inside the row's dimensions_json (what the row says)
-- raw_value    = the cell text inside raw_value_json ($.value, e.g. "-").
--                Falls back to the whole column if it has no "value" key.
prepared AS (
    SELECT *,
        regexp_extract(table_name, '([0-9]{4})', 1)                          AS table_year,
        regexp_extract_all(dimensions_json, '(20[0-9]{2})', 1)               AS years_in_row,
        coalesce(get_json_object(raw_value_json, '$.value'), raw_value_json) AS raw_value
    FROM all_tables
)

-- Summarize per table.
SELECT
    table_name,
    category,
    table_year                                       AS budget_year,
    expected_unit,

    -- ---------- Basic integrity ----------
    COUNT(*)                                         AS total_rows,
    -- Same record_id appearing more than once (technical duplicates)
    COUNT(*) - COUNT(DISTINCT record_id)             AS duplicate_ids,
    -- Rows with no dimension labels (no sector/region/year), cannot be interpreted
    SUM(CASE WHEN dimensions_json IS NULL OR trim(dimensions_json) IN ('', '{}') THEN 1 ELSE 0 END) AS missing_dimensions,

    -- ---------- Value quality: what is inside each cell? ----------
    -- Cells where no number could be read. This is the "no readable number" pile.
    -- The next three columns open that pile and say what was written in those cells:
    --   rows_raw_blank + rows_dash + rows_other_text = unparsed_values
    SUM(CASE WHEN parsed_numeric IS NULL THEN 1 ELSE 0 END) AS unparsed_values,
    -- ...of which the cell was empty
    SUM(CASE WHEN parsed_numeric IS NULL
              AND (raw_value IS NULL OR trim(raw_value) IN ('', 'null')) THEN 1 ELSE 0 END) AS rows_raw_blank,
    -- ...of which the cell was exactly a dash (source reports no amount; not zero, not blank)
    SUM(CASE WHEN parsed_numeric IS NULL
              AND trim(raw_value) IN ('-', '–', '—') THEN 1 ELSE 0 END) AS rows_dash,
    -- ...of which the cell had other text (a label, an error, etc.)
    SUM(CASE WHEN parsed_numeric IS NULL
              AND raw_value IS NOT NULL
              AND trim(raw_value) NOT IN ('', 'null', '-', '–', '—') THEN 1 ELSE 0 END) AS rows_other_text,
    -- Cells that were read as a real zero (a reported zero, different from a dash)
    SUM(CASE WHEN parsed_numeric = 0 THEN 1 ELSE 0 END) AS rows_zero,
    -- PDF text that wrapped onto a new line, in the cell value or the labels
    -- (checks a real newline and a JSON-escaped \n)
    SUM(CASE WHEN instr(raw_value, '\n') > 0 OR instr(raw_value_json, '\\n') > 0
               OR instr(dimensions_json, '\\n') > 0 THEN 1 ELSE 0 END) AS broken_pdf_text,
    -- A % sign inside the cell value: expected only in share tables, a leak if in amount tables
    SUM(CASE WHEN instr(raw_value, '%') > 0 THEN 1 ELSE 0 END) AS percentage_leaks,

    -- ---------- Totals and controls ----------
    -- Rows labelled total/subtotal/grand/control. Must be separated from detail rows
    -- before summing, otherwise amounts are double counted.
    SUM(CASE WHEN dimensions_json RLIKE '(?i)total|subtotal|grand|control' THEN 1 ELSE 0 END) AS subtotal_rows,

    -- ---------- Year check (important) ----------
    -- Every year found in the table's rows
    array_sort(array_distinct(flatten(collect_list(years_in_row)))) AS years_found,
    -- Distinct year combinations per row (shows if extra years sit on separate rows or share a row)
    array_distinct(collect_list(years_in_row))                      AS year_patterns,
    -- Rows that mention the table's year
    SUM(CASE WHEN array_contains(years_in_row, table_year) THEN 1 ELSE 0 END) AS rows_with_table_year,
    -- Rows that mention any OTHER year. For sector tables these are the later-year columns
    -- (Program / Proposed), so it is a flag to verify against the source, not automatically an error.
    SUM(CASE WHEN size(array_except(years_in_row, array(table_year))) > 0 THEN 1 ELSE 0 END) AS year_mismatch,
    -- PASS = only the table year appears; REVIEW = other years present, verify against source
    CASE WHEN SUM(CASE WHEN size(array_except(years_in_row, array(table_year))) > 0 THEN 1 ELSE 0 END) = 0
         THEN 'PASS' ELSE 'REVIEW' END                              AS year_check,

    -- ---------- Unit checks ----------
    -- Regional values above 999,999,999 thousand PHP = more than PHP 1 trillion.
    -- A 1,000x unit error is the costliest mistake, but values this big are more likely
    -- national or grand totals, so verify the source page before calling them errors.
    SUM(CASE WHEN category = 'Regional' AND parsed_numeric > 999999999 THEN 1 ELSE 0 END) AS unit_value_anomalies,
    -- Shares above 100 would mean a bad parse or a total mixed into detail rows
    SUM(CASE WHEN expected_unit LIKE 'Percentage%' AND parsed_numeric > 100 THEN 1 ELSE 0 END) AS shares_above_100_percent

FROM prepared
GROUP BY table_name, category, table_year, expected_unit
ORDER BY table_name;

## Current Profiling Summary

- **Sector 2024 tables** (`sector_2024_sector_amounts`, `sector_2024_sector_shares`)
  - 81 rows in each table (27 budget lines × 3 years).
  - No duplicate record IDs, no rows missing their labels, and no broken text from the PDF.
  - All 81 rows have a number. No dashes, blanks or zeros.
  - Three years appear: 2024, 2025 and 2026, with 27 rows each.
  - The year 2025 also appears in the Sector 2025 tables, so make sure to only base on the second table for 2025 values.
  - 6 rows in each table have labels such as "total" or "subtotal". These are summary rows, not budget lines.
  - No percentage share is above 100, so none looks like a misread amount.

- **Sector 2025 tables** (`sector_2025_sector_amounts`, `sector_2025_sector_shares`)
  - 81 rows in each table (27 budget lines × 3 years).
  - No duplicate record IDs, no rows missing their labels, and no broken text from the PDF.
  - All 81 rows have a number. No dashes, blanks or zeros.
  - Three years appear: 2025, 2026 and 2027, with 27 rows each.
  - 6 rows in each table have labels such as "total" or "subtotal". These are summary rows, not budget lines.
  - No percentage share is above 100, so none looks like a misread amount.

- **Regional 2024 table** (`regional_2024_regional_cells`)
  - 840 raw cells.
  - No duplicate record IDs, no rows missing their labels, and no broken text from the PDF.
  - 262 cells have no number. All 262 are dashes ("-"), meaning the source reported no amount. A dash is not zero and not blank. No cells are empty and none are zero.
  - Only the year 2024 appears.
  - 61 rows have labels such as "total" or "subtotal". These are summary rows, not regional detail.
  - 7 values are above 1 trillion thousand PHP. They are most likely national or grand totals, not unit errors.

- **Regional 2025 table** (`regional_2025_regional_cells`)
  - 960 raw cells.
  - No duplicate record IDs, no rows missing their labels, and no broken text from the PDF.
  - 336 cells have no number. All 336 are dashes ("-"), meaning the source reported no amount. A dash is not zero and not blank. No cells are empty and none are zero.
  - Only the year 2025 appears.
  - 67 rows have labels such as "total" or "subtotal". These are summary rows, not regional detail.
  - 7 values are above 1 trillion thousand PHP. They are most likely national or grand totals, not unit errors.

## Silver Rule Proposal

- **Sector 2024 tables** (`sector_2024_sector_amounts`, `sector_2024_sector_shares`)
  - Keep only the 2024 rows. These are the actual values.
  - Drop the 2025 and 2026 rows. They are Program and Proposed values.
  - Read the year from each row's `year` field, not from the table name.
  - Store the year as an integer, amounts as DECIMAL, shares as DECIMAL with two decimals, and labels as trimmed text.
  - Keep amounts (million PHP) and percentage shares as separate measures.
  - Separate summary rows (totals) from budget lines before adding anything up.

- **Sector 2025 tables** (`sector_2025_sector_amounts`, `sector_2025_sector_shares`)
  - Keep only the 2025 rows. These are the actual values.
  - Drop the 2026 and 2027 rows. They are Program and Proposed values.
  - Take 2025 only from this table, never from the Sector 2024 table.
  - Store the year as an integer, amounts as DECIMAL, shares as DECIMAL with two decimals, and labels as trimmed text.
  - Keep amounts (million PHP) and percentage shares as separate measures.
  - Separate summary rows (totals) from budget lines before adding anything up.

- **Regional 2024 table** (`regional_2024_regional_cells`)
  - Each row is a single cell from the PDF, not a complete budget record. Do not add up or count the 840 rows as if they were budget records.
  - Store dashes ("-") as NULL in the amount column. Do not turn them into zero.
  - Add a `value_state` column (for example `number` or `dash`) so a NULL from a dash can be told apart from any other missing value.
  - Store the year as an integer, amounts as DECIMAL, and labels as trimmed text.
  - Amounts are in thousand PHP. Convert to other units only on purpose, in a clearly named column.
  - Keep national totals separate from regional rows.
  - Separate summary rows (totals) from regional detail before adding anything up.

- **Regional 2025 table** (`regional_2025_regional_cells`)
  - Each row is a single cell from the PDF, not a complete budget record. Do not add up or count the 960 rows as if they were budget records.
  - Store dashes ("-") as NULL in the amount column. Do not turn them into zero.
  - Add a `value_state` column (for example `number` or `dash`) so a NULL from a dash can be told apart from any other missing value.
  - Store the year as an integer, amounts as DECIMAL, and labels as trimmed text.
  - Amounts are in thousand PHP. Convert to other units only on purpose, in a clearly named column.
  - Keep national totals separate from regional rows.
  - Separate summary rows (totals) from regional detail before adding anything up.

## Status

- **Sector 2024 tables**: Ready for Silver rule definition
- **Sector 2025 tables**: Ready for Silver rule definition
- **Regional 2024 table**: Review what the best course of action is for dash-filled cells. Ruling is currently set to transform it as null.
- **Regional 2025 table**: Review what the best course of action is for dash-filled cells. Ruling is currently set to transform it as null.

No Silver cleaning, normalization, joins, or analytical calculations have been performed during profiling.